# 03. Supervised Fine-Tuning on Large Language Model(Mistral-7B)

## 1. Install all packages

In [ ]:
import os
import getpass
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"  # Arrange GPU devices starting from 0
os.environ["CUDA_VISIBLE_DEVICES"]= "1"  # Set the GPU 1 to use
os.environ["HUGGING_FACE_HUB_TOKEN"] = getpass.getpass("Token:") #setup your tokens
assert os.environ["HUGGING_FACE_HUB_TOKEN"]

In [4]:
#!pip install -U transformers bitsandbytes accelerate sentencepiece peft -qqq

## 2. Import pretrained Tokenizer

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

## 3. Load the model with the quantization method

In [6]:
'''
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    trust_remote_code=True
  )
'''

# OR

from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
model.get_memory_footprint()/1000000000

4.551360512

In [8]:
def gen_function(prompt):
    from transformers import pipeline
    # 1) Query
    input_text = prompt

    # 2) options
    generator = pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
    )

    # 3) generation
    output = generator(
        input_text,
        max_length=500,
        do_sample=True,
        temperature=0.7
    )
    return output[0]['generated_text']
    

'''
def gen_function(prompt):
    # 1) Prompt
    input_text = prompt
    # 2) Tokenizing and Tensor transformation
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    # 3) Generate texts
    max_length = 100
    sample_outputs = model.generate(input_ids, do_sample=True, max_length=max_length, temperature=0.7)
    # 4) Decoding texts
    print(tokenizer.decode(sample_outputs[0], skip_special_tokens=True))
'''

'\ndef gen_function(prompt):\n    # 1) Prompt\n    input_text = prompt\n    # 2) Tokenizing and Tensor transformation\n    input_ids = tokenizer.encode(input_text, return_tensors="pt")\n\n    # 3) Generate texts\n    max_length = 100\n    sample_outputs = model.generate(input_ids, do_sample=True, max_length=max_length, temperature=0.7)\n    # 4) Decoding texts\n    print(tokenizer.decode(sample_outputs[0], skip_special_tokens=True))\n'

In [9]:
gen_function("Can you tell me the capital of Korea?")

/home/ktlim/anaconda3/envs/LLM/lib/python3.9/site-packages/transformers/generation/utils.py:1473: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"Can you tell me the capital of Korea?\n\n_Dave:_ Seoul.\n\n_Viet:_ And the capital of Japan?\n\n_Dave:_ Tokyo.\n\n_Viet:_ And the capital of China?\n\n_Dave:_ Beijing.\n\n_Viet:_ And the capital of Thailand?\n\n_Dave:_ Bangkok.\n\n_Viet:_ And the capital of Malaysia?\n\n_Dave:_ Kuala Lumpur.\n\n_Viet:_ And the capital of Indonesia?\n\n_Dave:_ Jakarta.\n\n_Viet:_ And the capital of Philippines?\n\n_Dave:_ Manila.\n\n_Viet:_ And the capital of Vietnam?\n\n_Dave:_ Hanoi.\n\n_Viet:_ And the capital of Singapore?\n\n_Dave:_ Singapore.\n\n_Viet:_ And the capital of Myanmar?\n\n_Dave:_ Yangon.\n\n_Viet:_ And the capital of Cambodia?\n\n_Dave:_ Phnom Penh.\n\n_Viet:_ And the capital of Laos?\n\n_Dave:_ Vientiane.\n\n_Viet:_ And the capital of Taiwan?\n\n_Dave:_ Taipei.\n\n_Viet:_ And the capital of Hong Kong?\n\n_Dave:_ Hong Kong.\n\n_Viet:_ And the capital of Macau?\n\n_Dave:_ Macau.\n\n_Viet:_ And the capital of Brunei?\n\n_Dave:_ Bandar Seri Begawan.\n\n_Viet:_ And the capital of Timor Les

## 04. Instruction tuning using Guanaco on LoRA

### 04-1. Data Load

In [10]:
from datasets import load_dataset

In [11]:
dataset_name = "timdettmers/openassistant-guanaco"
dataset = load_dataset(dataset_name, split="train[:1000]")

/home/ktlim/anaconda3/envs/LLM/lib/python3.9/site-packages/huggingface_hub/repocard.py:105: UserWarning: Repo card metadata block was not found. Setting CardData to empty.
  warnings.warn("Repo card metadata block was not found. Setting CardData to empty.")


In [12]:
dataset[1]

{'text': '### Human: ¿CUales son las etapas del desarrollo y en qué consisten según Piaget?### Assistant: Jean Piaget fue un psicólogo suizo que propuso una teoría sobre el desarrollo cognitivo humano que consta de cuatro etapas:\n\nEtapa sensoriomotora (0-2 años): Durante esta etapa, el niño aprende a través de sus sentidos y movimientos. Descubre que sus acciones pueden tener un impacto en el entorno y comienza a formarse una idea básica de objetividad y continuidad.\n\nEtapa preoperatoria (2-7 años): En esta etapa, el niño comienza a desarrollar un pensamiento simbólico y a comprender que las cosas pueden representar a otras cosas. También comienzan a desarrollar un pensamiento lógico y a comprender conceptos como la causa y el efecto.\n\nEtapa de operaciones concretas (7-12 años): Durante esta etapa, el niño desarrolla un pensamiento lógico y comprende las relaciones causales. Empiezan a comprender que las cosas pueden tener múltiples perspectivas y que los conceptos pueden ser más

In [13]:
def tokenize_function(examples):
	output = tokenizer(
      examples["text"],
      padding="max_length",
      truncation=True,
      max_length=200)
	return output

In [14]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

In [15]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
data_collator

DataCollatorForLanguageModeling(tokenizer=LlamaTokenizerFast(name_or_path='mistralai/Mistral-7B-v0.1', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}, mlm=False, mlm_probability=0.15, pad_to_multiple_of=None, tf_experimental_compile=False, return_tensors='pt')

### 04-2. LoRA Configuration

In [16]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )

In [17]:

from peft import LoraConfig, TaskType, get_peft_model

lora_alpha = 6
lora_dropout = 0.2
r = 6
target_module = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
    ]

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=r,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_module
)

In [18]:
peft_config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=6, target_modules={'up_proj', 'gate_proj', 'v_proj', 'o_proj', 'k_proj', 'q_proj', 'down_proj'}, lora_alpha=6, lora_dropout=0.2, fan_in_fan_out=False, bias='none', modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={})

### 04-3. Load model with LoRA

In [19]:
model = get_peft_model(model=model, peft_config=peft_config)

In [20]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): Linear4bit(
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.2, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=6, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=6, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
              )
              (k_proj): Linear4bit(
                (lora_dropout): ModuleDict(
  

In [21]:
model.print_trainable_parameters()

trainable params: 15,728,640 || all params: 7,257,460,736 || trainable%: 0.21672373536903142


In [22]:
#Save only for the LoRA adaptor
model_dir = "./mistral-7b-out"
model.save_pretrained(model_dir) 

In [23]:
from transformers import Trainer, TrainingArguments

per_device_train_batch_size = 4
gradient_accumulation_steps = 4
optim = "paged_adamw_32bit"
save_steps = 10
out_dir = "./mistral-7b-output"

train_arg = TrainingArguments(
    output_dir=out_dir,
    num_train_epochs=1,
    per_device_eval_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    fp16=True,
    group_by_length=True
)

In [24]:
max_seq_length = 512

trainer = Trainer(
    model=model,
    args=train_arg,
    train_dataset=tokenized_datasets,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [25]:
trainer.train()

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss


TrainOutput(global_step=31, training_loss=1.4391235843781502, metrics={'train_runtime': 107.3422, 'train_samples_per_second': 9.316, 'train_steps_per_second': 0.289, 'total_flos': 8483253151334400.0, 'train_loss': 1.4391235843781502, 'epoch': 0.99})

In [26]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Wed Nov 22 23:45:16 2023       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.113.01             Driver Version: 535.113.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A6000               On  | 00000000:01:00.0 Off |                  Off |
| 30%   28C    P8              28W / 300W |      5MiB / 49140MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [27]:
gen_function("What is the capital of Korea?")

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'LlamaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MvpForCausalLM', 'OpenLlamaForCausalLM', 'OpenAIGPTLMHeadModel', 'OPTForCausalLM', 'PegasusForCausalLM', 'PersimmonForCausalLM', 'PLBartFo

'What is the capital of Korea?\n\nSeoul\nSeoul is the capital city of South Korea, a country in East Asia.\n\nWhy is Seoul the capital of Korea?\n\nSeoul was chosen as the capital of Korea because of its location in the center of the country.\n\nWhat is the capital city of South Korea?\n\nSeoul\nSeoul is the capital of South Korea.\n\nWhat is the capital of North Korea?\n\nPyongyang\nThe capital of North Korea is Pyongyang, which is located in the North Korean province of Hwanghae-bukto.\n\nWhat is the capital of Korea?\n\nSeoul\nSeoul is the capital of South Korea; Pyongyang is the capital of North Korea.\n\nIs Seoul the capital of Korea?\n\nSeoul is the capital and largest city of South Korea. It is located in the northwest of the country, bordering the Yellow Sea, and is home to more than 10 million people.\n\nWhat is the capital city of Korea?\n\nSeoul\nSeoul is the capital and largest city of South Korea. It is located in the northwest of the country, bordering the Yellow Sea, and

## 05. Save and load trained lora model in huggingface

### 05-1. Save lora models

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [28]:
model.push_to_hub("my_mistral_lora") ## upload only lora adaptor

adapter_model.safetensors:   0%|          | 0.00/63.0M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/jujbob/my_mistral_lora/commit/4dbe5035c7f27b32d6ed60a5d449ce313bd3a21b', commit_message='Upload model', commit_description='', oid='4dbe5035c7f27b32d6ed60a5d449ce313bd3a21b', pr_url=None, pr_revision=None, pr_num=None)

### 05-2. Load lora models

In [1]:
import torch
from peft import PeftConfig, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast

peft_model_name = "jujbob/my_mistral_lora"
config = PeftConfig.from_pretrained(peft_model_name)
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='mistralai/Mistral-7B-v0.1', revision=None, task_type='CAUSAL_LM', inference_mode=True, r=6, target_modules={'q_proj', 'o_proj', 'gate_proj', 'v_proj', 'up_proj', 'down_proj', 'k_proj'}, lora_alpha=6, lora_dropout=0.2, fan_in_fan_out=False, bias='none', modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={})

In [2]:
backbone_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path
)

/home/ktlim/anaconda3/envs/LLM/lib/python3.9/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/ktlim/anaconda3/envs/LLM/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [5]:
#tokenizer = PreTrainedTokenizerFast.from_pretrained(config.base_model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

In [6]:
model = PeftModel.from_pretrained(backbone_model, peft_model_name)

In [7]:
def gen_function(prompt):
    from transformers import pipeline
    # 1) Query
    input_text = prompt

    # 2) options
    generator = pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
    )

    # 3) generation
    output = generator(
        input_text,
        max_length=500,
        do_sample=True,
        temperature=0.7
    )
    return output[0]['generated_text']

In [8]:
gen_function("What is the capital of Korea?")

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'LlamaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MvpForCausalLM', 'OpenLlamaForCausalLM', 'OpenAIGPTLMHeadModel', 'OPTForCausalLM', 'PegasusForCausalLM', 'PersimmonForCausalLM', 'PLBartFo

KeyboardInterrupt: 